# Introducción a los algoritmos evolutivos

En este notebook se ilustra la optimización con algoritmos evolutivos.

**Universidad Nacional de Colombia**

Departamento de Ciencias de la Computación y de la Decisión

Profesor: Juan David Ospina Arango
Monitor: Andrés Mauricio Zapata

**Introducción a Redes Neuronales y Algoritmos Bioinspirados**

Semestre 2026-02

## Planteamiento

En este notebook se muestra como una población inicial $P^{(0)}$ evoluciona para que los individuos que la componen minimicen la funcion $f_{fitness}$, que es una función de valor real.

Se representa por $x^{(t)}_{i} = [x^{(t)}_{i,1},\ldots,x^{(t)}_{i,d}]^T$ la $i$-ésima solución de la población $P^{(t)} \in \mathbb{R}^{n \times d}$.

A continuación se definirá cada uno de los pasos de un algoritmo evolutivo general.

## Generación de la población inicial



In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def pob_inicial(N = 30, d = 5,  LB = None, UB = None):

    """Genera una población inicial.

    Parameters
    ----------
    N : int
        Tamaño de la población
    d : int
        Dimensión del problema
    LB : float, vector de dimensión d
        Límite inferior de cada una de las variables del problema
    UB : float, vector de dimensión d
        Límite superior de cada una de las variables del problema

    Returns
    -------
    float
        Matriz de dimensión N (filas) por d (columnas)
    """
    if LB is None:
      LB = - np.ones(d)
    if UB is None:
      UB = np.ones(d)

    Pob_ini = np.random.rand(N, d)


    # Normalización
    Pob_ini = Pob_ini * (UB - LB)  + LB

    return Pob_ini


In [ ]:
# Prueba de escritorio:
np.random.seed(28)
A = pob_inicial(N = 20, d = 3, LB=np.array([-1,5,-8]), UB=np.array([10,15,-6]))
A

## Operación de mutación

A continuación se muestra la operación de mutación. Si $x^{(t)}_{i} = [x^{(t)}_{i,1},\ldots,x^{(t)}_{i,d}]^T$ es seleccionado para mutar y se decide que mute en la coordenada

In [ ]:
def mutacion(Pob, id_mutantes = None, coord_mut = None, LB = None, UB = None):

  """Genera los individuos mutantes

  Parámetros
  ----------

  Pob: float, matriz
      Cada elemento de la matriz Pob es un individuo que mutará.
  id_mutantes: int, vector
      Índices de los individuos que mutarán. Su dimensión es el número de filas
      de Pob.
  coord_mut: int, vector
      Coordenadas en las que se producirán las mutaciones.
  semilla: int
      Semilla para el generador de números aleatorios.
  """

  Pob = Pob.copy() # Para no afectar la matriz de entrada.

  (n_row, n_col) = Pob.shape

  if id_mutantes is None:
    id_mutantes = range(n_row)

  if coord_mut is None:
    coord_mut = np.random.randint(low = 0, high = n_col, size = n_row)

  if LB is None:
    LB = Pob.min(0) # mínimos de cada columna


  if UB is None:
    UB = Pob.max(0) # máximos de cada columna

  for i in id_mutantes:
    mutacion = np.random.rand() * (UB[coord_mut[i]] - LB[coord_mut[i]]) + LB[coord_mut[i]]
    Pob[i, coord_mut[i]] = mutacion

  return Pob[id_mutantes,:]


In [ ]:
# Prueba de escritorio:
Mutante = mutacion(Pob = A)

In [ ]:
Mutante

## Cruzamiento

A continuación se define la función de cruzamiento:

In [ ]:
def cruzamiento(Pob_cruce):

  """Cruzamiento

  Parámetros:
  ----------

  Pob_cruce: float, matriz
          Matriz con los individuos que se cruzarán
  semilla: int
          Semilla para el generador de números aleatorios
  """

  Pob_cruce = Pob_cruce.copy()

  (n_row, n_col) = Pob_cruce.shape

  Pob_hijos = np.zeros_like(Pob_cruce)
  n_cruzamiento = (np.floor(n_row/2)).astype(int)

  for i in range(n_cruzamiento):
    coord_cruzamiento = np.random.randint(low = 0, high = n_col, size = 1)[0]
    Pob_hijos[i,:] = np.concatenate((Pob_cruce[i, :coord_cruzamiento],Pob_cruce[i + n_cruzamiento, coord_cruzamiento:]))
    Pob_hijos[i + n_cruzamiento,:] = np.concatenate((Pob_cruce[i + n_cruzamiento, :coord_cruzamiento],Pob_cruce[i, coord_cruzamiento:]))
  return Pob_hijos




In [ ]:
# Prueba de escritorio:
Cruce = cruzamiento(A)

In [ ]:
Cruce

A continuación se define la función que toma una generación de soluciones y a partir de las operaciones genéticas construye la siguiente generación:

In [ ]:
def next_gen(Pob, fr_elitismo = 0.2, fr_mutacion = 0.1, LB = None, UB = None):

  """nex_gen

  Parámetros:
  -----------
  Pob: float, matriz
      Población de partida ordenada por el valor de la función objetivo
  fr_elitismo: float, (entre 0 y 1)
      Porcentaje de elitismo
  fr_mutacion: float, (entre 0 y 1)
      Porcentaje de individuos que muta
  LB: float, vector
      Límite inferior de cada una de las variables de decisión
  UB: float, vector
      Límite superior de cada una de las variables de decisión

  """


  Pob = Pob.copy()
  (n_row, n_col) = Pob.shape

  n_mutacion = (np.floor(n_row * fr_mutacion)).astype(int)
  n_cruzamiento = (2 * np.floor(n_row*(1 - fr_elitismo - fr_mutacion)/2)).astype(int)
  n_elite = n_row - n_mutacion - n_cruzamiento

  # Población élite
  Pob_elite = Pob[:n_elite,]

  # Población cruce
  indices_cruzamiento = np.random.choice(n_row, size=n_cruzamiento, replace=False)
  Pob_cruce = cruzamiento(Pob[indices_cruzamiento,])

  # Población mutante:
  id_mutantes = np.random.randint(low = 0, high = n_col, size = n_mutacion)
  Pob_mutante = mutacion(Pob, id_mutantes = id_mutantes, LB = LB, UB = UB)

  # Nueva población
  Pob_next_gen = np.concatenate((Pob_elite, Pob_cruce, Pob_mutante), axis = 0)
  return (Pob_next_gen)


In [ ]:
# Prueba de escritorio:
B = next_gen(A,fr_elitismo = 0.2, fr_mutacion = 0.12)


In [ ]:
B

Evaluación de la población con la función objetivo y ranking de las soluciones:

In [ ]:
def pob_eval(Pob, f_obj):

  """pob_eval

  Parámetros:
  ----------
  Pob: float, matriz
      Población a evaluar. Cada fila es una solución
  f_obj: function
      Función objetivo


  Returns:
  --------
  Pob_ranked: float, matriz
    Población rankeada. La primera fila tiene el mejor valor de la función objetivo (el más bajo)

  f_best: float
    Mejor valor de la función objetivo (el más pequeño)


  """

  Pob = Pob.copy()
  (n_row, n_col) = Pob.shape
  fitness_val = np.zeros((n_row))

  # Evaluación de la función objetivo en la población
  for i in range(n_row):
    fitness_val[i] = f_obj(Pob[i,:])

  # Ranking de las soluciones
  pop_ranking = np.argsort(fitness_val)

  # Extracción del mejor valor en la población
  f_best = fitness_val[pop_ranking[0]]

  return(Pob[pop_ranking], f_best)

Algoritmo evolutivo con todo lo anterior:

In [ ]:
def mi_algoritmo_evolutivo(f_obj, N = 30, d = 3, LB = None, UB = None, fr_elitismo = 0.2, fr_mutacion = 0.10, max_iter = 50, semilla = None):

  """mi_algoritmo_evolutivo

  Parámetros:
  ----------

    f_obj: function
        Función objetivo
    N : int
        Tamaño de la población
    d : int
        Dimensión del problema
    LB : float, vector de dimensión d
        Límite inferior de cada una de las variables del problema
    UB : float, vector de dimensión d
        Límite superior de cada una de las variables del problema
    fr_elitismo: float, (entre 0 y 1)
        Porcentaje de elitismo
    fr_mutacion: float, (entre 0 y 1)
        Porcentaje de individuos que muta
    max_iter: int
        Máximo número de iteraciones
    semilla : int
        Semilla para el generador de números aleatorios


  Returns:
  --------
  Pob: list of floats,
    Población en cada generación

  f_best: float
    Mejor valor de la función objetivo (el más pequeño) en cada generación


  """

  Pob0 = pob_inicial(N = N, d = d,  LB = LB, UB = UB)

  Pob = []
  f_best = []
  Pob.append(Pob0)

  if not semilla is None:
    np.random.seed(semilla)

  #print(np.random.get_state()[1][0])
  for i in range(max_iter):
    Pob_ranked, f_best_gen_i = pob_eval(Pob = Pob[i], f_obj = f_obj)
    f_best.append(f_best_gen_i)
    Pob.append(next_gen(Pob_ranked, fr_elitismo = fr_elitismo, fr_mutacion = fr_mutacion, LB = LB, UB = UB))
  return(Pob,np.array(f_best))



## Ejemplo: optimización de una función en 2D

A continuación se muestra la optimización de la función

$f(x) = \|x \|_2$

### Caso unimodal

A continuación se muestra la optimización de la función

$f(x) = \|x \|_2$

In [ ]:
# Definición de la función:
def parabola(x):
  return(np.sqrt(np.sum(x**2)))

In [ ]:
# Optimización
Poblacion, mejor_solucion = mi_algoritmo_evolutivo(f_obj = parabola, d = 2, LB = np.array([-5,-5]), UB = np.array([5,5]),fr_elitismo = 0.1, max_iter=1000, semilla = 20)

In [ ]:
# Gráfica del valor de la función objetivo de la mejor solución en cada generación.
plt.plot(np.arange(len(mejor_solucion)),mejor_solucion)
plt.title('Mejor valor de f_obj')
plt.grid()
plt.show()

Evolución de algunas generaciones:

In [ ]:
for j in np.arange(200,220):
  fig, ax = plt.subplots()
  plt.scatter(Poblacion[j][:,0], Poblacion[j][:,1])
  plt.xlim([-5,5])
  plt.ylim([-5,5])
  plt.grid()
  plt.title('Generación ' + str(j))
  plt.show()

### Caso multimodal

A continuación se muestra la optimización de la función $f(x) = \sum_{i=1}^ p{x_i^2 + \cos{(6x_i)}}$, $-1 \leq x_i \leq 1$, $i=1, \ldots, p$.

In [ ]:
# Definición de la función:
def f(x):
  y = x**2 * np.cos(6*x)
  return(y)

def mi_f(x):
  y = [f(t) for t in x]
  y = np.array(y).sum()
  return(y)

In [ ]:
# Paso intermedio para graficar las curvas de nivel de la función:
n_points = 50
X = np.linspace(-1,1,n_points)
Y = np.linspace(-1,1,n_points)
X, Y = np.meshgrid(X,Y)

In [ ]:
# Paso intermedio para graficar las curvas de nivel de la función. Evaluación de la función:
Z = np.array([mi_f(np.array([X[i,j],Y[i,j]])) for i in range(n_points) for j in range(n_points)]).reshape(n_points,n_points)

In [ ]:
# Gráfica de las curvas de nivel de la función:
fig, ax = plt.subplots()
ax.contour(X,Y,Z)
plt.xlim([-1,1])
plt.ylim([-1,1])
plt.grid()
plt.show()

Optimización de la función

In [ ]:
Poblacion, mejor_solucion = mi_algoritmo_evolutivo(f_obj = mi_f, d = 2,fr_elitismo = 0.1, max_iter=100, semilla = 2)

In [ ]:
# Gráfica del valor de la función objetivo de la mejor solución en cada generación.
plt.plot(np.arange(len(mejor_solucion)),mejor_solucion)
plt.title('Mejor valor de f_obj')
plt.grid()
plt.show()

In [ ]:
fig, ax = plt.subplots(subplot_kw={"projection": "3d"})
ax.plot_surface(X,Y,Z)
plt.show()

In [ ]:
# Gráfica del valor de la función objetivo de la mejor solución en cada generación.


for j in range(100):
  fig, ax = plt.subplots()
  ax.contour(X,Y,Z)
  ax.scatter(Poblacion[j][:,0], Poblacion[j][:,1])
  plt.xlim([-1,1])
  plt.ylim([-1,1])
  plt.title('Generación ' + str(j))
  plt.show()

## Con PyGad

A continuación se presenta la optimización con [PyGad](https://pygad.readthedocs.io/en/latest/pygad.html#pygad-ga-class).



In [ ]:
# Instalación de la librería:
%%capture
!pip install pygad

In [ ]:
# Carga de la librería
import pygad
import numpy as np
import matplotlib.pyplot as plt


### Problema multimodal

Se usará primero la [función de Rastrigin](https://en.wikipedia.org/wiki/Rastrigin_function) para ilustrar la optimización.

In [ ]:
# Función de Rastrigin

def rastrigin(x):
  val = 10 * len(x)

  for i in x:
    val = val + (i**2 - 10 * np.cos(np.pi * i))

  return(val)

In [ ]:
# Pasos intermedios para graficar las curvas de nivel de la función:

n_points = 100
X = np.linspace(-5.12,5.12,n_points)
Y = np.linspace(-5.12,5.12,n_points)
X, Y = np.meshgrid(X,Y)


In [ ]:
Z = np.array([rastrigin(np.array([X[i,j],Y[i,j]])) for i in range(n_points) for j in range(n_points)]).reshape(n_points,n_points)

In [ ]:
fig, ax = plt.subplots()
ax.contour(X,Y,Z, levels = 25)
plt.title('Curvas de nivel de la función de Rastrigin')
plt.xlim([-5.12,5.12])
plt.ylim([-5.12,5.12])
plt.show()



A continuación se define la función de aptitud



In [ ]:
# Para usar PyGad la función de fitness debe aceptar tres argumentos:
#  - ga_instance: el objeto instanciado a partir de la clase PyGad
#  - solution: la solución que se desea evaluar en la función objetivo
#  - solution_id: no se toca, pero corresponde al identificador de la solución evaluada en la población actual

def fitness(ga_instancia,solucion,solucion_idx):
  output = rastrigin(solucion)
  if not np.all((solucion>= -5.12) & (solucion<= 5.12)):
    return(-10000)

  fitness = -output
  return(fitness)

In [ ]:
# Prueba de escritorio 1:
fitness(None, np.array([0.1, 1.5]), None)

In [ ]:
# Prueba de escritorio 2:
rastrigin(np.array([0.1, 1.5]))

A continuación se parametriza el algoritmo evolutivo. Esto se hace instanciando un objeto de la clase `PyGad`. Primero se definen los parámetros:

In [ ]:
fitness_function = fitness

num_generations = 500
num_parents_mating = 2 # En la reproducción sexual cuántos individuos intervienen

sol_per_pop = 30
num_genes = 2

init_range_low = -5.12
init_range_high = 5.12


parent_selection_type = "random"
keep_parents = 0
keep_elitism = 5

crossover_type = "single_point"

mutation_type = "random"
mutation_num_genes=1
# mutation_percent_genes = 1

Ahora se instancia la clase `PyGad`:

In [ ]:
mi_f_ga_instance = pygad.GA(num_generations=num_generations,
                       num_parents_mating=num_parents_mating,
                       fitness_func=fitness_function,
                       sol_per_pop=sol_per_pop,
                       num_genes=num_genes,
                       init_range_low=init_range_low,
                       init_range_high=init_range_high,
                       parent_selection_type=parent_selection_type,
                       keep_parents=keep_parents,
                       keep_elitism = keep_elitism,
                       crossover_type=crossover_type,
                       mutation_type=mutation_type,
                       mutation_num_genes=mutation_num_genes,
                       random_seed=1983,
                       save_solutions=True)

In [ ]:
mi_f_ga_instance.run()

Veamos el resultado:

In [ ]:
solution, solution_fitness, solution_idx = mi_f_ga_instance.best_solution()
print("Parameters of the best solution : {solution}".format(solution=solution))
print("Fitness value of the best solution = {solution_fitness}".format(solution_fitness=solution_fitness))

In [ ]:
Pob = (np.array(mi_f_ga_instance.solutions))

In [ ]:
for j in range(0,50):
  Pob = np.array(mi_f_ga_instance.solutions[j*10:(j+1)*10])
  fig, ax = plt.subplots()
  ax.contour(X,Y,Z)
  ax.scatter(Pob[:,0], Pob[:,1])
  plt.xlim([-5.12,5.12])
  plt.ylim([-5.12,5.12])
  plt.title('Generación ' + str(j))
  plt.show()

### El problema de la mochila (Knapsack Problem)

A continuación se ilustra el uso de `PyGad` para resolver [el problema de la mochila](https://en.wikipedia.org/wiki/Knapsack_problem).


#### Modalidad 0/1

En esta modalidad en la vitrina están exhibidos los ítems de los cuales solo hay una unidad de cada uno.

In [ ]:
# Problem Parameters
items = [
    {"name": "Laptop", "weight": 3, "value": 2000},
    {"name": "Smartphone", "weight": 1, "value": 1500},
    {"name": "Camera", "weight": 2, "value": 1200},
    {"name": "Headphones", "weight": 1, "value": 300},
    {"name": "Tablet", "weight": 2, "value": 1000},
    {"name": "Power Bank", "weight": 1, "value": 200},
]

weights = [item["weight"] for item in items]
values = [item["value"] for item in items]

knapsack_capacity = 5

In [ ]:
np.sum(np.array([1,1,0,1,0,1])*weights)

In [ ]:
np.sum(np.array([1,1,0,1,0,1])*values)

Definición de la función de aptitud:

In [ ]:
# Fitness Function (la función objetivo va por dentro)
def fitness_func(ga_instancia,solution,solucion_idx):
  '''
  Computes the fitness function.
  solution vector of 1 and 0 of length n.
  solution_idx: is the solution identifier.
  n is the number of items.
  '''
  # Calculate total weight and value of selected items
  total_weight = np.sum(solution * weights)
  total_value = np.sum(solution * values)
  # Penalize solutions that exceed the knapsack capacity
  if total_weight > knapsack_capacity:
      return -10
  return total_value

Parametrización del algoritmo:

In [ ]:
# Number of genes = number of items (0 or 1 for each item)
num_genes = len(items)

# Create an instance of the GA class
ga_instance = pygad.GA(
    num_generations=600,
    num_parents_mating=2,
    num_genes=6,
    sol_per_pop = 60,
    init_range_low=0,
    init_range_high=2,  # Since we want binary values (0 or 1)
    gene_type= int,
    fitness_func=fitness_func,
    keep_elitism = 2,
    crossover_type = "single_point",
    mutation_type="random",
    mutation_num_genes=1,
    random_seed=1983
)


Ejecución del algoritmo:

In [ ]:
# Run the genetic algorithm
ga_instance.run()

Extracción de la mejor solución:

In [ ]:
# Get the best solution
solution, solution_fitness, _ = ga_instance.best_solution()
#solution = solution.astype(int)  # Convert to integers (0 or 1)



In [ ]:
print(solution)

In [ ]:
# Print results
print("\nKnapsack Problem Results:")
print("-" * 50)
print(f"Knapsack capacity: {knapsack_capacity} units")
print("\nBest solution found:")
print("-" * 50)
total_weight = 0
total_value = 0

print("Selected items:")
for i, (selected, item) in enumerate(zip(solution, items)):
    if selected:
        print(f"- {item['name']}: Weight = {item['weight']}, Value = ${item['value']}")
        total_weight += item['weight']
        total_value += item['value']

print("-" * 50)
print(f"Total Weight: {total_weight}/{knapsack_capacity} units")
print(f"Total Value: ${total_value}")



# Actividad
1. ¿En cuántas generaciones se alcanzó el máximo?
2. ¿Cuál es la mejor solución para las siguientes capacidades de la mochila? a) W = 4.5, b) W = 4, c) W = 3.5 y d) W = 3.
3. ¿Qué tienen en común estas soluciones?


#### Modalidad k/n

Hay que tomar k elementos de n disponibles para cada unidad. Ya no pensamos en una vitrina sino en un almacen.

In [ ]:
# Parametrizació del problema
items = [
    {"name": "Laptop", "weight": 3, "value": 2000, "units": 5},
    {"name": "Smartphone", "weight": 1, "value": 1500, "units": 3},
    {"name": "Camera", "weight": 2, "value": 1200, "units": 16},
    {"name": "Headphones", "weight": 1, "value": 300, "units": 4},
    {"name": "Tablet", "weight": 2, "value": 1000, "units": 5},
    {"name": "Power Bank", "weight": 1, "value": 200, "units": 20},
]

weights = [item["weight"] for item in items]
values = [item["value"] for item in items]
units = [item["units"] for item in items]

knapsack_capacity = 15

Definición de una función de normalización para poder usar la codificación en números reales para las soluciones del problema:

In [ ]:
# Esta función normaliza una solución para que todos sus elementos sean
# números enteros no negativos.
def sol_trans(x):
  x = np.maximum(x, 0)
  x = x.astype('int')
  return(x)

In [ ]:
# Prueba de escritorio de la función de normalización:
x1 = np.array([-0.15, 2.3, 9.89, -0.003])
sol_trans(x1)

In [ ]:
x2 = np.array([-300, 2.9, 9.0001, -4])
sol_trans(x2)

In [ ]:
sol_trans(x1)==sol_trans(x2)

Función de aptitud:

In [ ]:
# Fitness Function
def fitness_func(ga_instance, solution, solution_idx):
    # Calculate total weight and value of selected items
    solution = sol_trans(solution) # Normalización de la solución.
    total_weight = np.sum(solution * weights)
    total_value = np.sum(solution * values)

    # Check if solution exceeds knapsack capacity or available units
    if total_weight > knapsack_capacity:
        return -20

    # Check if solution exceeds available units
    for i, count in enumerate(solution):
        if count > units[i]:
            return -20

    # Return fitness value
    return total_value

Parametrización del algoritmo evolutivo:

In [ ]:
# Number of genes = number of items (0 or 1 for each item)
num_genes = len(items)

# Create an instance of the GA class
ga_instance = pygad.GA(
    num_generations=5000,
    num_parents_mating=2,
    num_genes=6,
    sol_per_pop = 60,
    init_range_low=np.zeros(6),
    init_range_high=np.array(units, dtype =float)+1,  #
    #gene_type=int,
    fitness_func=fitness_func,
    keep_parents = 1,
    keep_elitism = 2,
    crossover_type = "two_points",
    mutation_probability = 0.1,
    mutation_type="random",
    random_seed=8
)

In [ ]:
# Run the genetic algorithm
ga_instance.run()

In [ ]:
# Get the best solution
solution, solution_fitness, _ = ga_instance.best_solution()
solution = sol_trans(solution)
print(f"La mejor solución es {solution} y su valor objetivo correspondiente es {solution_fitness}.")

In [ ]:
# Print results
print("\nKnapsack Problem Results:")
print("-" * 50)
print(f"Knapsack capacity: {knapsack_capacity} units")
print("\nBest solution found:")
print("-" * 50)
total_weight = 0
total_value = 0

print("Selected items:")
for i, (selected, item) in enumerate(zip(solution, items)):
    if selected:
        print(f"- {item['name']}: Weight = {item['weight']}, Value = ${item['value']}, Cant. = {selected}")
        total_weight += item['weight'] * selected
        total_value += item['value'] * selected

print("-" * 50)
print(f"Total Weight: {total_weight}/{knapsack_capacity} unitsof weight")
print(f"Total Value: ${total_value}")

In [ ]:
ga_instance.plot_fitness()
plt.show()

# Actividad
1. ¿Puede mejorar la solución? ¿Qué se puede hacer para explorarlo? (Hacerlo).
2. Parametrice este algoritmo con una codificación de números enteros. ¿Qué diferencias vemos en la convergencia?
3. ¿Qué otros parámetros parecen mejorar la convergencia en este caso?